In [7]:
# 1. INSTALL
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("roboflow", "ultralytics")

# 2. IMPORT
import os, json, shutil, time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from ultralytics import YOLO
from PIL import Image
from roboflow import Roboflow

# 3. KONFIGURASI
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

ROBOFLOW_API_KEY = secrets.get_secret("ROBOFLOW_API_KEY")

# Proyek 1: dataset ikan tunggal
WORKSPACE_1  = secrets.get_secret("WORKSPACE")
PROJECT_1    = secrets.get_secret("PROJECT_1")
VERSION_1    = int(secrets.get_secret("VERSION_1"))

# Proyek 2: dataset ikan berkelompok
WORKSPACE_2  = secrets.get_secret("WORKSPACE")
PROJECT_2    = secrets.get_secret("PROJECT_2")
VERSION_2    = int(secrets.get_secret("VERSION_2"))

# Path model
YOLO_MODEL_PATH   = "/kaggle/input/models/ahmadfarhanh/yolo/pytorch/default/1/best (1).pt"
EFFNET_MODEL_PATH = "/kaggle/input/models/ahmadfarhanh/efficient-fish/pytorch/default/1/efficientnet_best.pth"

# Output
OUTPUT_DIR        = Path("/kaggle/working/hasil_evaluasi")
MERGED_TEST_DIR   = Path("/kaggle/working/merged_test")

# Hyperparameter pipeline
YOLO_CONF_THRESHOLD          = 0.417
EFFNET_PESSIMISTIC_THRESHOLD = 0.65
CLASS_NAMES = ["Segar", "Tidak_Segar"]

# Mapping class index YOLO → label kesegaran untuk ground truth
# 0: Fresh-Eye, 1: Fresh-Skin, 2: NonFresh-Eye, 3: NonFresh-Skin
FRESH_CLASS_IDS    = {0, 1}   # SEGAR
NONFRESH_CLASS_IDS = {2, 3}   # TIDAK SEGAR

# 4. DOWNLOAD DATASET
def download_test_set(api_key, workspace, project_name, version, dest_dir: Path) -> Path:
    print(f"\n[INFO] Downloading {workspace}/{project_name} v{version} ...")
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(workspace).project(project_name)
    dataset = project.version(version).download(
        model_format="yolov8",
        location=str(dest_dir),
        overwrite=True
    )
    test_images = Path(dataset.location) / "test" / "images"

    if not test_images.exists():
        raise FileNotFoundError(
            f"Folder test/images tidak ditemukan di: {dataset.location}"
        )
    n = len(list(test_images.glob("*.*")))
    print(f"[INFO] Test set: {n} gambar ditemukan di {test_images}")
    return Path(dataset.location)

# 5. MERGE TEST SET

def merge_test_sets(dataset_dirs: list[Path], merged_dir: Path):
    merged_images = merged_dir / "images"
    merged_labels = merged_dir / "labels"
    merged_images.mkdir(parents=True, exist_ok=True)
    merged_labels.mkdir(parents=True, exist_ok=True)

    total = 0
    for ds_dir in dataset_dirs:
        src_images = ds_dir / "test" / "images"
        src_labels = ds_dir / "test" / "labels"
        prefix = ds_dir.name

        for img_path in sorted(src_images.glob("*.*")):
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                continue

            new_stem = img_path.stem
            dest_img = merged_images / img_path.name
            if dest_img.exists():
                new_stem = f"{prefix}_{img_path.stem}"
                dest_img = merged_images / f"{new_stem}{img_path.suffix}"

            shutil.copy2(img_path, dest_img)

            src_lbl = src_labels / f"{img_path.stem}.txt"
            if src_lbl.exists():
                shutil.copy2(src_lbl, merged_labels / f"{new_stem}.txt")

            total += 1

    print(f"\n[INFO] Merge selesai — {total} gambar di {merged_dir}")

# 6. GROUND TRUTH DARI FILE LABEL

def parse_ground_truth_from_label(label_path: Path) -> str | None:
    if not label_path.exists():
        return None

    lines = label_path.read_text().strip().splitlines()
    if not lines:
        return None

    class_ids = set()
    for line in lines:
        parts = line.strip().split()
        if parts:
            try:
                class_ids.add(int(parts[0]))
            except ValueError:
                continue

    if class_ids & NONFRESH_CLASS_IDS:
        return "TIDAK SEGAR"
    if class_ids & FRESH_CLASS_IDS:
        return "SEGAR"
    return None

# 7. LOAD MODEL

def load_models(yolo_path: str, effnet_path: str, device: torch.device):
    print(f"\n[INFO] Memuat YOLOv8        : {yolo_path}")
    model_yolo = YOLO(yolo_path)

    print(f"[INFO] Memuat EfficientNet-B3 : {effnet_path}")
    model_effnet = models.efficientnet_b3(weights=None)
    num_ftrs = model_effnet.classifier[1].in_features
    model_effnet.classifier[1] = nn.Linear(num_ftrs, len(CLASS_NAMES))
    model_effnet.load_state_dict(torch.load(effnet_path, map_location=device))
    model_effnet.to(device)
    model_effnet.eval()

    return model_yolo, model_effnet


img_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 8. LOGIKA INFERENSI

def run_pipeline(image_path: Path, model_yolo, model_effnet, device) -> dict:
    """Menjalankan full two-stage pipeline pada satu gambar."""
    filename = image_path.name

    try:
        original_img = Image.open(image_path).convert("RGB")
    except Exception as e:
        return {"filename": filename, "error": str(e), "status_kesimpulan": "ERROR"}

    # Stage 1: YOLOv8
    results = model_yolo.predict(
        source=original_img,
        conf=YOLO_CONF_THRESHOLD,
        save=False,
        verbose=False
    )
    result = results[0]

    if len(result.boxes) == 0:
        return {
            "filename": filename,
            "status_kesimpulan": "TIDAK TERDETEKSI",
            "total_fitur_terdeteksi": 0,
            "detail_deteksi": [],
            "warning": None
        }

    # Cek duplikasi organ (indikasi multi-ikan)
    base_features = []
    for box in result.boxes:
        raw = model_yolo.names[int(box.cls[0])].lower()
        organ = (raw
                 .replace("nonfresh", "").replace("fresh", "")
                 .replace("tidak_segar", "").replace("segar", "")
                 .replace("_", "").strip())
        base_features.append(organ)

    warning_msg = None
    if len(base_features) != len(set(base_features)):
        warning_msg = (
            "Duplikasi organ terdeteksi, kemungkinan lebih dari 1 ikan dalam frame. "
            "Akurasi inferensi dapat menurun."
        )

    # Stage 2: EfficientNet-B3
    is_non_fresh = False
    details = []

    for i, box in enumerate(result.boxes):
        yolo_class_name = model_yolo.names[int(box.cls[0])]
        yolo_conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        cropped = original_img.crop((x1, y1, x2, y2))
        tensor  = img_transforms(cropped).unsqueeze(0).to(device)

        with torch.no_grad():
            probs          = F.softmax(model_effnet(tensor), dim=1)
            eff_conf_t, preds = torch.max(probs, 1)
            eff_conf       = float(eff_conf_t[0].item())
            eff_class      = CLASS_NAMES[preds[0].item()]

        status_effnet = eff_class
        if eff_class == "Tidak_Segar":
            if eff_conf >= EFFNET_PESSIMISTIC_THRESHOLD:
                is_non_fresh = True
            else:
                status_effnet = "Tidak_Segar (Dianulir)"

        details.append({
            "id_fitur"                : i + 1,
            "yolo_class"              : yolo_class_name,
            "yolo_confidence"         : round(yolo_conf * 100, 2),
            "efficientnet_prediction" : status_effnet,
            "efficientnet_confidence" : round(eff_conf * 100, 2),
            "koordinat"               : {"x1": x1, "y1": y1, "x2": x2, "y2": y2}
        })

    return {
        "filename"              : filename,
        "warning"               : warning_msg,
        "status_kesimpulan"     : "TIDAK SEGAR" if is_non_fresh else "SEGAR",
        "total_fitur_terdeteksi": len(result.boxes),
        "detail_deteksi"        : details,
    }

# 9. METRIK

def compute_metrics(all_results: list[dict]) -> dict:
    tp = fp = fn = tn = 0
    skipped = not_detected = 0

    for r in all_results:
        gt   = r.get("ground_truth")
        pred = r.get("status_kesimpulan")

        if gt is None:
            skipped += 1
            continue

        if pred in ("TIDAK TERDETEKSI", "ERROR"):
            not_detected += 1
            # Gagal deteksi → anggap prediksi salah
            if gt == "TIDAK SEGAR":
                fn += 1
            else:
                fp += 1
            continue

        if   gt == "TIDAK SEGAR" and pred == "TIDAK SEGAR": tp += 1
        elif gt == "TIDAK SEGAR" and pred == "SEGAR":        fn += 1
        elif gt == "SEGAR"       and pred == "TIDAK SEGAR":  fp += 1
        elif gt == "SEGAR"       and pred == "SEGAR":        tn += 1

    total = tp + fp + fn + tn
    if total == 0:
        return {"error": "Tidak ada gambar yang bisa dievaluasi."}

    precision = tp / (tp + fp)       if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn)       if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    accuracy  = (tp + tn) / total

    return {
        "total_gambar_dievaluasi" : total,
        "gambar_tanpa_gt"         : skipped,
        "gambar_tidak_terdeteksi" : not_detected,
        "confusion_matrix"        : {"TP": tp, "FP": fp, "FN": fn, "TN": tn},
        "precision"               : round(precision, 4),
        "recall"                  : round(recall,    4),
        "f1_score"                : round(f1,        4),
        "accuracy"                : round(accuracy,  4),
    }

# MAIN

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MERGED_TEST_DIR.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Device: {device}")

    # Download dari Roboflow
    ds1_dir = download_test_set(
        ROBOFLOW_API_KEY, WORKSPACE_1, PROJECT_1, VERSION_1,
        Path("/kaggle/working/dataset1")
    )
    ds2_dir = download_test_set(
        ROBOFLOW_API_KEY, WORKSPACE_2, PROJECT_2, VERSION_2,
        Path("/kaggle/working/dataset2")
    )

    # Merge test set
    merge_test_sets([ds1_dir, ds2_dir], MERGED_TEST_DIR)

    merged_images = MERGED_TEST_DIR / "images"
    merged_labels = MERGED_TEST_DIR / "labels"

    image_paths = sorted([
        p for p in merged_images.glob("*.*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ])
    print(f"[INFO] Total gambar siap dievaluasi: {len(image_paths)}")

    # Load model
    model_yolo, model_effnet = load_models(YOLO_MODEL_PATH, EFFNET_MODEL_PATH, device)

    # Inferensi
    all_results = []
    start = time.time()

    for idx, img_path in enumerate(image_paths, 1):
        result = run_pipeline(img_path, model_yolo, model_effnet, device)

        label_path = merged_labels / f"{img_path.stem}.txt"
        result["ground_truth"] = parse_ground_truth_from_label(label_path)

        all_results.append(result)

        gt_label = result["ground_truth"] or "?"
        status   = result["status_kesimpulan"]
        match    = "✓" if status == result["ground_truth"] else ("?" if not result["ground_truth"] else "✗")

        print(f"[{idx:>4}/{len(image_paths)}] {img_path.name:<45} "
              f"GT: {gt_label:<12} | Pred: {status:<14} {match}")

    elapsed = time.time() - start

    # Cetak metrik
    metrics = compute_metrics(all_results)

    print("\n" + "=" * 58)
    print("   HASIL EVALUASI END-TO-END TWO-STAGE PIPELINE")
    print("=" * 58)
    print(f"  Waktu total          : {elapsed:.1f}s "
          f"({elapsed / len(image_paths) * 1000:.0f} ms/gambar)")
    print(f"  Total dievaluasi     : {metrics.get('total_gambar_dievaluasi', '-')}")
    print(f"  Gambar tanpa GT      : {metrics.get('gambar_tanpa_gt', '-')}")
    print(f"  Tidak terdeteksi     : {metrics.get('gambar_tidak_terdeteksi', '-')}")

    if "confusion_matrix" in metrics:
        cm = metrics["confusion_matrix"]
        print(f"\n  Confusion Matrix  (positif = TIDAK SEGAR)")
        print(f"    TP  busuk terdeteksi benar  : {cm['TP']}")
        print(f"    TN  segar terdeteksi benar  : {cm['TN']}")
        print(f"    FP  segar salah → dibuang   : {cm['FP']}")
        print(f"    FN  busuk lolos → berbahaya : {cm['FN']}")
        print(f"\n  Precision : {metrics['precision']:.4f}")
        print(f"  Recall    : {metrics['recall']:.4f}")
        print(f"  F1-Score  : {metrics['f1_score']:.4f}")
        print(f"  Accuracy  : {metrics['accuracy']:.4f}")
    else:
        print(f"\n  [!] {metrics.get('error')}")

    print("=" * 58)

    # Simpan output
    output = {
        "konfigurasi": {
            "proyek_1"                     : f"{WORKSPACE_1}/{PROJECT_1} v{VERSION_1}",
            "proyek_2"                     : f"{WORKSPACE_2}/{PROJECT_2} v{VERSION_2}",
            "yolo_conf_threshold"          : YOLO_CONF_THRESHOLD,
            "effnet_pessimistic_threshold" : EFFNET_PESSIMISTIC_THRESHOLD,
            "total_gambar"                 : len(image_paths),
            "waktu_total_detik"            : round(elapsed, 2),
        },
        "metrik"            : metrics,
        "detail_per_gambar" : all_results,
    }

    out_json = OUTPUT_DIR / "hasil_evaluasi.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"\n[INFO] Hasil lengkap    → {out_json}")

    errors = [
        r for r in all_results
        if r.get("ground_truth")
        and r["status_kesimpulan"] not in ("TIDAK TERDETEKSI", "ERROR")
        and r["status_kesimpulan"] != r["ground_truth"]
    ]
    if errors:
        err_json = OUTPUT_DIR / "gambar_salah.json"
        with open(err_json, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)
        print(f"[INFO] {len(errors)} salah klasifikasi → {err_json}")
    else:
        print("[INFO] Tidak ada kesalahan klasifikasi.")


if __name__ == "__main__":
    main()

[INFO] Device: cuda

[INFO] Downloading ahmad-farhan-hidayat-s-workspace/fish-freshness-0by5o-nqtfg v3 ...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/dataset1 in yolov8:: 100%|██████████| 9479/9479 [00:18<00:00, 503.01it/s] 


[INFO] Test set: 474 gambar ditemukan di /kaggle/working/dataset1/test/images

[INFO] Downloading ahmad-farhan-hidayat-s-workspace/fish-freshness-yqy4n-ggiyd v1 ...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/dataset2 in yolov8:: 100%|██████████| 351/351 [00:00<00:00, 6283.61it/s]


[INFO] Test set: 3 gambar ditemukan di /kaggle/working/dataset2/test/images

[INFO] Merge selesai — 477 gambar di /kaggle/working/merged_test
[INFO] Total gambar siap dievaluasi: 477

[INFO] Memuat YOLOv8        : /kaggle/input/models/ahmadfarhanh/yolo/pytorch/default/1/best (1).pt
[INFO] Memuat EfficientNet-B3 : /kaggle/input/models/ahmadfarhanh/efficient-fish/pytorch/default/1/efficientnet_best.pth
[   1/477] IMG_0172_jpeg.rf.15c87e1ab8483d4572dcf4b985e4f894.jpg GT: SEGAR        | Pred: SEGAR          ✓
[   2/477] IMG_0207_jpeg.rf.f43249ac48137fa193ad031e1a139c4e.jpg GT: SEGAR        | Pred: SEGAR          ✓
[   3/477] IMG_0213_jpeg.rf.2e29a38a7ba9401f4bfcf81597c7df7b.jpg GT: SEGAR        | Pred: SEGAR          ✓
[   4/477] IMG_0217_jpeg.rf.002158415b80df657e33613d48bd9460.jpg GT: SEGAR        | Pred: SEGAR          ✓
[   5/477] IMG_0266_jpeg.rf.3cd7f6df54c1990e46a532d3db8c577c.jpg GT: SEGAR        | Pred: SEGAR          ✓
[   6/477] IMG_0274_jpeg.rf.465e60811cbdf79cfc0668c54f3a8a6c.